## ANOVA 테스트

In [5]:
import pandas as pd
import numpy as np
from collections import Counter
from scipy import stats
from scipy.stats import chi2_contingency, f_oneway
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 0. 데이터 로드
# ============================================================
DATA_PATH   = r'데이터\8,9번 파일(최종)\M19_도매_소매업(최종).parquet'
OUTPUT_BASE = r'20번. 기업 생애주기'   # 출력 경로 (필요 시 수정)

df = pd.read_parquet(DATA_PATH)
print(f"전체 데이터: {df.shape[0]:,}행 × {df.shape[1]}컬럼")
print(f"회계년도 범위: {df['회계년도'].min()} ~ {df['회계년도'].max()}")
print(f"부실: {df['부실라벨_ICR3년'].sum():,}  |  정상: {(df['부실라벨_ICR3년']==0).sum():,}")


# ============================================================
# 1. 분석 대상 컬럼 자동 추출
#    · float64 전체에서 생애주기 분류·식별자·라벨 등 비분석 컬럼만 제외
# ============================================================
# 분석 제외 컬럼 (ID / 라벨 / 생애주기 원재료 / 범주형)
EXCLUDE_COLS = {
    '사업자등록번호', '회계년도', '회사명', '종업원',
    '통계청 한국표준산업분류 코드 11차(대분류)',
    '통계청 한국표준산업분류 11차(중분류)',
    '부실라벨_ICR3년',
    'M코드', '회사명_norm',
    # 생애주기 OCF/ICF/FCF 부호 원본
    '영업현금흐름비율',
    '투자활동으로 인한 현금흐름(요약)(백만원)',
    '재무활동으로 인한 현금흐름(요약)(백만원)',
    # 금액 원본(비율 아님)
    '자산총계(요약)(백만원)',
    # 특성·더미 변수
    '빅4감사', '업력',
}

ALL_ANALYSIS_COLS = [
    c for c in df.select_dtypes(include='float64').columns
    if c not in EXCLUDE_COLS
]
print(f"\n분석 대상 컬럼: {len(ALL_ANALYSIS_COLS)}개")


# ============================================================
# 2. 컬럼을 의미별로 자동 그루핑 (출력 가독성용)
# ============================================================
def auto_group(col):
    suffixes = ['_diff_industry', '_ratio_industry', '_diff', '_ratio']
    for sfx in suffixes:
        if col.endswith(sfx):
            return {
                '_diff_industry': '산업대비_변화량(diff_industry)',
                '_ratio_industry': '산업대비_비율(ratio_industry)',
                '_diff':           '전기대비_변화량(diff)',
                '_ratio':          '전기대비_비율(ratio)',
            }[sfx]
    return '기본 재무비율'

# 그룹 딕셔너리: {그룹명: [컬럼명, ...]}
from collections import defaultdict
GROUPS = defaultdict(list)
for col in ALL_ANALYSIS_COLS:
    GROUPS[auto_group(col)].append(col)

GROUP_ORDER = [
    '기본 재무비율',
    '전기대비_변화량(diff)',
    '전기대비_비율(ratio)',
    '산업대비_변화량(diff_industry)',
    '산업대비_비율(ratio_industry)',
]
for g in GROUP_ORDER:
    print(f"  {g}: {len(GROUPS.get(g, []))}개")


# ============================================================
# 3. Dickinson(2011) 생애주기 분류
# ============================================================
def classify_lifecycle(ocf_sign, icf_sign, fcf_sign):
    lifecycle_map = {
        '+/-/-': '성숙기',
        '+/-/+': '성장기',
        '-/-/+': '도입기',
        '-/+/-': '쇠퇴기',
        '-/+/+': '쇠퇴기',
        '+/+/-': None,   # 쇄신기 → 제외
        '-/-/-': None,
        '+/+/+': None,
    }
    return lifecycle_map.get(f"{ocf_sign}/{icf_sign}/{fcf_sign}", None)


df['OCF_부호'] = df['영업현금흐름비율'].apply(lambda x: '+' if x > 0 else '-')
df['ICF_부호'] = df['투자활동으로 인한 현금흐름(요약)(백만원)'].apply(lambda x: '+' if x > 0 else '-')
df['FCF_부호'] = df['재무활동으로 인한 현금흐름(요약)(백만원)'].apply(lambda x: '+' if x > 0 else '-')
df['생애주기_단년'] = df.apply(
    lambda row: classify_lifecycle(row['OCF_부호'], row['ICF_부호'], row['FCF_부호']),
    axis=1
)


# ============================================================
# 4. 최근 3년 다수결로 생애주기 확정
# ============================================================
def get_majority_lifecycle(group):
    valid = group.sort_values('회계년도')
    valid = valid[valid['생애주기_단년'].notna()]
    if valid.empty:
        return None
    recent = valid.tail(3)
    last   = valid.iloc[-1]['생애주기_단년']
    if len(recent) < 2:
        return last
    counter = Counter(recent['생애주기_단년'].tolist())
    mc = counter.most_common()
    return mc[0][0] if mc[0][1] >= 2 else last


lifecycle_result = (
    df.groupby('사업자등록번호')
      .apply(get_majority_lifecycle)
      .reset_index()
)
lifecycle_result.columns = ['사업자등록번호', '생애주기_최종']

last_rows = (
    df.sort_values('회계년도')
      .groupby('사업자등록번호')
      .last()
      .reset_index()
)
last_rows = last_rows.merge(lifecycle_result, on='사업자등록번호', how='left')
last_rows = last_rows[last_rows['생애주기_최종'].notna()].copy()

LIFECYCLE_ORDER = ['도입기', '성장기', '성숙기', '쇠퇴기']
last_rows['생애주기_최종'] = pd.Categorical(
    last_rows['생애주기_최종'], categories=LIFECYCLE_ORDER, ordered=True
)
last_rows = last_rows.sort_values('생애주기_최종')

n_total = len(last_rows)
n_bad   = int(last_rows['부실라벨_ICR3년'].sum())
n_good  = n_total - n_bad

print(f"\n=== 생애주기 분류 완료 (쇄신기 제외) ===")
cnt = last_rows['생애주기_최종'].value_counts()
for lc in LIFECYCLE_ORDER:
    print(f"  {lc}: {cnt.get(lc, 0):,}개  "
          f"(부실률 {last_rows[last_rows['생애주기_최종']==lc]['부실라벨_ICR3년'].mean()*100:.2f}%)")
print(f"\n전체 기업: {n_total:,}  |  부실: {n_bad:,}  |  정상: {n_good:,}")


# ============================================================
# 5. Winsorize (하위 1%, 상위 2%)
# ============================================================
def winsorize(series, lower=0.01, upper=0.02):
    lo = series.quantile(lower)
    hi = series.quantile(1 - upper)
    return series.clip(lower=lo, upper=hi)


df_w = last_rows.copy()
for col in ALL_ANALYSIS_COLS:
    if col in df_w.columns:
        df_w[col] = winsorize(df_w[col])

print(f"\n=== Winsorize 완료 (하위 1%, 상위 2%) — 대상: {len(ALL_ANALYSIS_COLS)}개 변수 ===")


# ============================================================
# 6. Scheffe 사후검정 함수
# ============================================================
def scheffe_test(groups_data, group_names, alpha=0.05):
    k         = len(groups_data)
    N         = sum(len(g) for g in groups_data)
    SSW       = sum(((g - g.mean()) ** 2).sum() for g in groups_data)
    df_within = N - k
    MSW       = SSW / df_within if df_within > 0 else np.nan

    rows = []
    for i in range(k):
        for j in range(i + 1, k):
            g1, g2    = groups_data[i], groups_data[j]
            n1, n2    = len(g1), len(g2)
            mean_diff = g1.mean() - g2.mean()

            denom = MSW * (1/n1 + 1/n2)
            if denom == 0 or np.isnan(denom):
                continue
            F_scheffe = (mean_diff ** 2) / denom
            p_approx  = 1 - stats.f.cdf(F_scheffe / (k - 1), k - 1, df_within)
            sig = ('***' if p_approx < 0.001 else
                   '**'  if p_approx < 0.01  else
                   '*'   if p_approx < 0.05  else 'n.s.')

            rows.append({
                '비교':        f"{group_names[i]} vs {group_names[j]}",
                '평균(A)':     round(g1.mean(), 4),
                '평균(B)':     round(g2.mean(), 4),
                '평균차(A-B)': round(mean_diff, 4),
                'F(Scheffe)':  round(F_scheffe, 4),
                'p값':         round(p_approx, 4),
                '유의성':      sig,
            })
    return pd.DataFrame(rows)


# ============================================================
# 7. 전체 컬럼 ANOVA + Scheffe 실행
# ============================================================
print(f"\n{'='*70}")
print(f"ANOVA + Scheffe 사후검정 — 전체 {len(ALL_ANALYSIS_COLS)}개 변수")
print(f"{'='*70}")

anova_summary_rows = []
scheffe_all        = []
skipped_cols       = []

for group_name in GROUP_ORDER:
    cols_in_group = GROUPS.get(group_name, [])
    if not cols_in_group:
        continue

    print(f"\n\n{'━'*70}")
    print(f"  [{group_name}]  ({len(cols_in_group)}개 변수)")
    print(f"{'━'*70}")

    for col in cols_in_group:
        if col not in df_w.columns:
            skipped_cols.append(col)
            continue

        # 생애주기별 그룹 구성
        groups, group_names, desc_rows = [], [], []
        for lc in LIFECYCLE_ORDER:
            sub = df_w[df_w['생애주기_최종'] == lc][col].dropna()
            if len(sub) >= 5:
                groups.append(sub.values)
                group_names.append(lc)
                desc_rows.append({
                    '생애주기': lc, 'N': len(sub),
                    '평균':     round(sub.mean(), 4),
                    '중위수':   round(sub.median(), 4),
                    '표준편차': round(sub.std(), 4),
                })

        if len(groups) < 2:
            skipped_cols.append(col)
            continue

        try:
            f_stat, p_value = f_oneway(*groups)
        except Exception:
            skipped_cols.append(col)
            continue

        all_vals   = np.concatenate(groups)
        grand_mean = all_vals.mean()
        ss_between = sum(len(g) * (g.mean() - grand_mean) ** 2 for g in groups)
        ss_total   = ((all_vals - grand_mean) ** 2).sum()
        eta_sq     = ss_between / ss_total if ss_total > 0 else np.nan
        eta_label  = ('대' if eta_sq >= 0.14 else
                      '중' if eta_sq >= 0.06 else '소')

        sig_mark = ('***' if p_value < 0.001 else
                    '**'  if p_value < 0.01  else
                    '*'   if p_value < 0.05  else 'n.s.')

        print(f"\n  ▶ {col}")
        print(pd.DataFrame(desc_rows).to_string(index=False))
        print(f"\n    F = {f_stat:.4f}  |  p = {p_value:.6f}  {sig_mark}"
              f"  |  Eta² = {eta_sq:.4f} ({eta_label} 효과)")

        if p_value < 0.05:
            sch = scheffe_test(groups, group_names)
            if not sch.empty:
                print(f"\n    [Scheffe 사후검정]")
                print(sch.to_string(index=False))
                sch.insert(0, '변수',   col)
                sch.insert(0, '변수그룹', group_name)
                scheffe_all.append(sch)
        else:
            print(f"    → ANOVA 비유의 → 사후검정 생략")

        anova_summary_rows.append({
            '변수그룹':  group_name,
            '변수':      col,
            'F통계량':   round(f_stat, 4),
            'p값':       round(p_value, 6),
            '유의성':    sig_mark,
            'Eta²':      round(eta_sq, 4) if not np.isnan(eta_sq) else np.nan,
            '효과크기':  eta_label,
        })


# ============================================================
# 8. ANOVA 요약 출력
# ============================================================
anova_summary = pd.DataFrame(anova_summary_rows)

print(f"\n\n{'='*70}")
print("ANOVA 요약 (전체 변수)")
print(f"{'='*70}")
print(anova_summary.to_string(index=False))

if skipped_cols:
    print(f"\n⚠ 스킵된 변수 ({len(skipped_cols)}개): {skipped_cols}")


# ============================================================
# 9. 부실위험 카이제곱 검정
# ============================================================
print(f"\n\n{'='*70}")
print("카이제곱 검정 — 생애주기별 부실위험 차이")
print(f"{'='*70}")

crosstab = pd.crosstab(
    last_rows['생애주기_최종'],
    last_rows['부실라벨_ICR3년'],
    margins=True
)
crosstab.columns   = ['정상', '부실', '합계']
crosstab['부실률(%)'] = (crosstab['부실'] / crosstab['합계'] * 100).round(2)
print("\n[교차표]")
print(crosstab.to_string())

ct_raw = pd.crosstab(last_rows['생애주기_최종'], last_rows['부실라벨_ICR3년'])
chi2, p_chi2, dof, _ = chi2_contingency(ct_raw)
cramers_v = np.sqrt(chi2 / (len(last_rows) * (min(ct_raw.shape) - 1)))

print(f"\n[전체 카이제곱 검정]")
print(f"  χ² = {chi2:.4f}  |  자유도 = {dof}  |  p = {p_chi2:.6f}"
      f"  {'***' if p_chi2 < 0.001 else '**' if p_chi2 < 0.01 else '*' if p_chi2 < 0.05 else 'n.s.'}")
print(f"  Cramér's V = {cramers_v:.4f}"
      f"  ({'강한' if cramers_v >= 0.3 else '중간' if cramers_v >= 0.1 else '약한'} 연관성)")

print(f"\n[생애주기별 개별 카이제곱]")
print(f"  {'구분':8s}  {'업체수(구성비)':18s}  {'부실률':8s}  {'χ²값':12s}  {'유의성'}")
print(f"  {'-'*64}")
for lc in LIFECYCLE_ORDER:
    sub  = last_rows[last_rows['생애주기_최종'] == lc]
    rest = last_rows[last_rows['생애주기_최종'] != lc]
    n, ratio = len(sub), len(sub) / n_total * 100
    bad_r = sub['부실라벨_ICR3년'].mean() * 100
    ct2 = pd.DataFrame({
        '해당':   [(sub['부실라벨_ICR3년'] == 0).sum(),  sub['부실라벨_ICR3년'].sum()],
        '나머지': [(rest['부실라벨_ICR3년'] == 0).sum(), rest['부실라벨_ICR3년'].sum()],
    })
    try:
        c2, p2, d2, _ = chi2_contingency(ct2)
        sig2 = '***' if p2 < 0.001 else '**' if p2 < 0.01 else '*' if p2 < 0.05 else 'n.s.'
        print(f"  {lc:8s}  {n:6,}({ratio:5.1f}%)      {bad_r:5.2f}%    {c2:10.2f}  {sig2}")
    except Exception as e:
        print(f"  {lc:8s}  계산 불가: {e}")


# ============================================================
# 10. 결과 저장
# ============================================================
anova_summary.to_csv(
    f'{OUTPUT_BASE}/lifecycle_anova_summary_full.csv',
    index=False, encoding='utf-8-sig'
)
crosstab.to_csv(
    f'{OUTPUT_BASE}/lifecycle_chisquare_crosstab.csv',
    encoding='utf-8-sig'
)
if scheffe_all:
    pd.concat(scheffe_all, ignore_index=True).to_csv(
        f'{OUTPUT_BASE}/lifecycle_scheffe_detail_full.csv',
        index=False, encoding='utf-8-sig'
    )

print(f"\n\n✅ 저장 완료")
print(f"   lifecycle_anova_summary_full.csv      : ANOVA 요약 (전체 변수, F값·p값·Eta²)")
print(f"   lifecycle_scheffe_detail_full.csv     : Scheffe 사후검정 쌍별 상세")
print(f"   lifecycle_chisquare_crosstab.csv      : 카이제곱 교차표")

전체 데이터: 39,908행 × 277컬럼
회계년도 범위: 2012 ~ 2024
부실: 1,509  |  정상: 38,399

분석 대상 컬럼: 262개
  기본 재무비율: 64개
  전기대비_변화량(diff): 72개
  전기대비_비율(ratio): 72개
  산업대비_변화량(diff_industry): 27개
  산업대비_비율(ratio_industry): 27개

=== 생애주기 분류 완료 (쇄신기 제외) ===
  도입기: 1,327개  (부실률 41.15%)
  성장기: 1,005개  (부실률 19.20%)
  성숙기: 2,524개  (부실률 11.33%)
  쇠퇴기: 1,173개  (부실률 36.66%)

전체 기업: 6,029  |  부실: 1,455  |  정상: 4,574

=== Winsorize 완료 (하위 1%, 상위 2%) — 대상: 262개 변수 ===

ANOVA + Scheffe 사후검정 — 전체 262개 변수


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  [기본 재무비율]  (64개 변수)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  ▶ 부채비율
생애주기    N        평균      중위수      표준편차
 도입기 1327 4030.5266 311.3088 8287.2425
 성장기 1005 1345.9004 187.5939 4579.0438
 성숙기 2524 1015.7414  95.9486 3888.6559
 쇠퇴기 1173 2708.8679 198.1200 6657.8975

    F = 88.5011  |  p = 0.000000  ***  |  Eta² = 0.0422 (소 효과)

    [Scheffe 사후검정]
        비교     평균(A)     평균(B)   평균차(A-B)  F(Scheffe)    p값  유의성
도입기 

In [2]:
import pandas as pd
import numpy as np
from collections import Counter

# ============================================================
# 0. 데이터 로드
# ============================================================
df = pd.read_parquet(r'데이터\8,9번 파일(최종)\M19_도매_소매업(최종).parquet')
print(f"전체 데이터: {df.shape[0]:,}행 × {df.shape[1]}컬럼")


# ============================================================
# 1. Dickinson 생애주기 분류 함수 (동일)
# ============================================================
def classify_lifecycle(ocf_sign, icf_sign, fcf_sign):
    pattern = f"{ocf_sign}/{icf_sign}/{fcf_sign}"
    lifecycle_map = {
        '+/-/-': '성숙기',
        '+/-/+': '성장기',
        '-/-/+': '도입기',
        '-/+/-': '쇠퇴기',
        '-/+/+': '쇠퇴기',
        '+/+/-': '조정기',
        '-/-/-': '조정기',
        '+/+/+': '조정기',
    }
    return lifecycle_map.get(pattern, '조정기')


# ============================================================
# 2. 부호 컬럼 미리 생성 (반복 불필요)
# ============================================================
df['OCF_부호'] = df['영업현금흐름비율'].apply(lambda x: '+' if x > 0 else '-')
df['ICF_부호'] = df['투자활동으로 인한 현금흐름(요약)(백만원)'].apply(lambda x: '+' if x > 0 else '-')
df['FCF_부호'] = df['재무활동으로 인한 현금흐름(요약)(백만원)'].apply(lambda x: '+' if x > 0 else '-')

df['생애주기_단년'] = df.apply(
    lambda row: classify_lifecycle(row['OCF_부호'], row['ICF_부호'], row['FCF_부호']),
    axis=1
)


# ============================================================
# 3. 최근 3년 다수결 생애주기 확정 함수 (동일)
# ============================================================
def get_majority_lifecycle(group):
    group_sorted = group.sort_values('회계년도')
    recent = group_sorted.tail(3)
    last_lifecycle = group_sorted.iloc[-1]['생애주기_단년']
    if len(recent) < 3:
        return last_lifecycle
    counter = Counter(recent['생애주기_단년'].tolist())
    most_common = counter.most_common()
    return most_common[0][0] if most_common[0][1] >= 2 else last_lifecycle


# ============================================================
# 4. 부실률 계산 함수 (WoE/IV 대신 단순 부실률)
# ============================================================
def calculate_bad_rate(df_input, category_col, label_col):
    rows = []
    for cat in df_input[category_col].unique():
        sub  = df_input[df_input[category_col] == cat]
        bad  = int(sub[label_col].sum())
        good = int((sub[label_col] == 0).sum())
        total = len(sub)
        rows.append({
            '생애주기': cat,
            '전체':     total,
            '부실':     bad,
            '정상':     good,
            '부실률(%)': round(bad / total * 100, 2),
        })
    result = pd.DataFrame(rows).sort_values('부실률(%)', ascending=False).reset_index(drop=True)
    return result


# ============================================================
# 5. 연도별 루프 (2015 ~ 2024)
# ============================================================
all_years_rate   = []   # 연도별 부실률 테이블 누적
all_years_scored = []   # 연도별 기업별 점수 누적

for year in range(2015, 2025):  # 2015 ~ 2024

    # ── 해당 연도까지 누적 데이터 ──────────────────────────────
    df_cut = df[df['회계년도'] <= year].copy()

    if df_cut.empty:
        print(f"\n[{year}] 데이터 없음, 스킵")
        continue

    # ── 생애주기 확정 ─────────────────────────────────────────
    lifecycle_result = (
        df_cut.groupby('사업자등록번호')
              .apply(get_majority_lifecycle)
              .reset_index()
    )
    lifecycle_result.columns = ['사업자등록번호', '생애주기_최종']

    # ── 마지막 행 추출 + 생애주기 병합 ───────────────────────
    last_rows = (
        df_cut.sort_values('회계년도')
              .groupby('사업자등록번호')
              .last()
              .reset_index()
    )
    last_rows = last_rows.merge(lifecycle_result, on='사업자등록번호', how='left')

    total_bad  = int(last_rows['부실라벨_ICR3년'].sum())
    total_good = int((last_rows['부실라벨_ICR3년'] == 0).sum())

    if total_bad == 0 or total_good == 0:
        print(f"\n[{year}] 부실/정상 한쪽이 0 → 계산 불가, 스킵")
        continue

    # ── 부실률 계산 ───────────────────────────────────────────
    rate_table = calculate_bad_rate(last_rows, '생애주기_최종', '부실라벨_ICR3년')
    rate_table.insert(0, '기준연도', year)

    # ── 부실률 softmax 변환 후 min-max 스케일링 (0~100) ──────
    # 1) softmax: exp(부실률) / sum(exp(부실률))
    #    (overflow 방지를 위해 max값을 빼고 exp 계산)
    rate_vals = rate_table['부실률(%)'].values*0.01
    exp_vals  = np.exp(rate_vals - rate_vals.max())
    softmax_vals = exp_vals / exp_vals.sum()

    # 2) softmax 결과를 0~100으로 min-max 스케일링
    sm_min = softmax_vals.min()
    sm_max = softmax_vals.max()
    if sm_max == sm_min:
        rate_table['위험점수_최종'] = 50.0   # 모두 동점이면 중간값
    else:
        rate_table['위험점수_최종'] = (
            (softmax_vals - sm_min) / (sm_max - sm_min) * 100
        ).round(1)

    # ── rate_table 누적 (★ 이 줄이 빠져있었습니다) ──────────
    all_years_rate.append(rate_table)

    # ── 기업별 점수 부여 ──────────────────────────────────────
    score_map = dict(zip(rate_table['생애주기'], rate_table['위험점수_최종']))
    last_rows['기준연도']    = year
    last_rows['생애주기_점수'] = last_rows['생애주기_최종'].map(score_map)
    all_years_scored.append(
        last_rows[['기준연도', '사업자등록번호', '회계년도',
                   '생애주기_최종', '생애주기_점수', '부실라벨_ICR3년']]
    )

    print(f"\n[{year}] 기업 {len(last_rows):,}개  |  부실 {total_bad:,}  |  정상 {total_good:,}")
    print(rate_table[['생애주기', '부실률(%)', '위험점수_최종']].to_string(index=False))


# ============================================================
# 6. 결과 합치기 & 저장
# ============================================================
result_rate    = pd.concat(all_years_rate,    ignore_index=True)
result_scored  = pd.concat(all_years_scored, ignore_index=True)

result_scored.to_csv(
    r'20번. 기업 생애주기\lifecycle_scored_yearly_minmax.csv',
    index=False
)
result_rate.to_csv(
    r'20번. 기업 생애주기\lifecycle_rate_yearly_minmax.csv',
    index=False, encoding='utf-8-sig'
)

print("\n✅ 저장 완료")
print("   - lifecycle_scored_yearly_minmax.parquet : 연도별 기업 점수")
print("   - lifecycle_rate_yearly_minmax.csv        : 연도별 생애주기 부실률/점수 테이블")

전체 데이터: 39,908행 × 277컬럼


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_22448\2523007927.py:96: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(get_majority_lifecycle)



[2015] 기업 3,140개  |  부실 489  |  정상 2,651
생애주기  부실률(%)  위험점수_최종
 쇠퇴기   29.46    100.0
 도입기   24.68     75.7
 조정기   15.62     32.8
 성숙기    8.87      3.2
 성장기    8.10      0.0


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_22448\2523007927.py:96: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(get_majority_lifecycle)



[2016] 기업 3,443개  |  부실 563  |  정상 2,880
생애주기  부실률(%)  위험점수_최종
 쇠퇴기   31.42    100.0
 도입기   27.44     80.6
 조정기   15.11     25.3
 성장기    9.01      0.3
 성숙기    8.93      0.0


...

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_22448\2523007927.py:96: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(get_majority_lifecycle)



[2022] 기업 6,123개  |  부실 1,188  |  정상 4,935
생애주기  부실률(%)  위험점수_최종
 쇠퇴기   34.80    100.0
 도입기   29.61     76.6
 조정기   19.55     34.7
 성장기   10.87      1.7
 성숙기   10.39      0.0


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_22448\2523007927.py:96: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(get_majority_lifecycle)



[2023] 기업 6,124개  |  부실 1,305  |  정상 4,819
생애주기  부실률(%)  위험점수_최종
 도입기   36.22    100.0
 쇠퇴기   35.45     96.6
 조정기   19.80     33.5
 성장기   13.62     11.1
 성숙기   10.40      0.0

[2024] 기업 6,124개  |  부실 1,509  |  정상 4,615
생애주기  부실률(%)  위험점수_최종
 도입기   42.87    100.0
 쇠퇴기   38.11     82.9
 조정기   22.36     31.7
 성장기   19.15     22.2
 성숙기   11.18      0.0

✅ 저장 완료
   - lifecycle_scored_yearly_minmax.parquet : 연도별 기업 점수
   - lifecycle_rate_yearly_minmax.csv        : 연도별 생애주기 부실률/점수 테이블


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_22448\2523007927.py:96: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(get_majority_lifecycle)
